In [12]:
# Basic Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder,
    FunctionTransformer
)

# Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Model
from sklearn.linear_model import LogisticRegression

# Train-Test Split and Cross Validation
from sklearn.model_selection import train_test_split, cross_val_score

# Save and Load Model
import joblib

# Load only the required columns
features = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "purpose",
    "addr_state",
    "dti",
    "delinq_2yrs",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "issue_d"
]

required_columns = features + ["loan_status"]

df = pd.read_csv(
    "Data/loan.csv",
    usecols=required_columns,
    low_memory=False
)

print("Complete Dataset Shape:", df.shape)

display(df.head())

Complete Dataset Shape: (2260668, 20)


,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,addr_state,dti,delinq_2yrs,open_acc,pub_rec,revol_bal,revol_util,total_acc
0,2500,36 months,13.56,84.92,C,10+ years,RENT,55000.0,Not Verified,Dec-2018,Current,debt_consolidation,NY,18.24,0.0,9.0,1.0,4341,10.3,34.0
1,30000,60 months,18.94,777.23,D,10+ years,MORTGAGE,90000.0,Source Verified,Dec-2018,Current,debt_consolidation,LA,26.52,0.0,13.0,1.0,12315,24.2,44.0
2,5000,36 months,17.97,180.69,D,6 years,MORTGAGE,59280.0,Source Verified,Dec-2018,Current,debt_consolidation,MI,10.51,0.0,8.0,0.0,4599,19.1,13.0
3,4000,36 months,18.94,146.51,D,10+ years,MORTGAGE,92000.0,Source Verified,Dec-2018,Current,debt_consolidation,WA,16.74,0.0,10.0,0.0,5468,78.1,13.0
4,30000,60 months,16.14,731.78,C,10+ years,MORTGAGE,57250.0,Not Verified,Dec-2018,Current,debt_consolidation,MD,26.35,0.0,12.0,0.0,829,3.6,26.0


In [13]:
# Keep only Fully Paid and Charged Off loans
df = df[
    df["loan_status"].isin(["Fully Paid", "Charged Off"])
].copy()

# Sample 15% of the dataset
df = df.sample(
    frac=0.15,
    random_state=42
)

# Convert target into binary values
df["loan_status"] = df["loan_status"].map({
    "Fully Paid": 0,
    "Charged Off": 1
})

# Separate input and target
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

print("Dataset Shape:", df.shape)

print("\nInput Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nTarget Distribution:")
print(y.value_counts())

Dataset Shape: (195541, 20)

Input Shape: (195541, 19)
Target Shape: (195541,)

Target Distribution:
loan_status
0    156014
1     39527
Name: count, dtype: int64


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

print("\nTraining Target Distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting Target Distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training Data: (156432, 19)
Testing Data: (39109, 19)

Training Target Distribution:
loan_status
0    79.786105
1    20.213895
Name: proportion, dtype: float64

Testing Target Distribution:
loan_status
0    79.784704
1    20.215296
Name: proportion, dtype: float64


In [15]:
# Dataset information
df.info()

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Missing values summary
print("\nMissing Values Summary:")

missing_data = df.isnull().sum()

for column in df.columns:
    if missing_data[column] > 0:
        print(column, ":", missing_data[column])

<class 'pandas.DataFrame'>
Index: 195541 entries, 1509134 to 943951
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   loan_amnt            195541 non-null  int64  
 1   term                 195541 non-null  str    
 2   int_rate             195541 non-null  float64
 3   installment          195541 non-null  float64
 4   grade                195541 non-null  str    
 5   emp_length           184235 non-null  str    
 6   home_ownership       195541 non-null  str    
 7   annual_inc           195541 non-null  float64
 8   verification_status  195541 non-null  str    
 9   issue_d              195541 non-null  str    
 10  loan_status          195541 non-null  int64  
 11  purpose              195541 non-null  str    
 12  addr_state           195541 non-null  str    
 13  dti                  195487 non-null  float64
 14  delinq_2yrs          195541 non-null  float64
 15  open_acc             195541

In [16]:
# Separate numeric columns
numeric_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

print("Numeric Columns:")
print(numeric_cols)

# KNNImputer demonstration on a small sample
knn_sample = X_train[numeric_cols].sample(
    n=min(5000, len(X_train)),
    random_state=42
)

knn_imputer = KNNImputer(n_neighbors=5)

knn_data = knn_imputer.fit_transform(knn_sample)

print("\nKNNImputer completed successfully")

print("Original Sample Shape:", knn_sample.shape)
print("KNN Imputed Shape:", knn_data.shape)

Numeric Columns:
['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc']

KNNImputer completed successfully
Original Sample Shape: (5000, 11)
KNN Imputed Shape: (5000, 11)


In [17]:
# Function to create new features
def create_features(data):

    data = data.copy()

    # Date decomposition
    data["issue_d"] = pd.to_datetime(
        data["issue_d"],
        format="%b-%Y",
        errors="coerce"
    )

    data["issue_year"] = data["issue_d"].dt.year
    data["issue_month"] = data["issue_d"].dt.month

    # Interaction feature
    data["income_loan_ratio"] = (
        data["annual_inc"] / data["loan_amnt"].replace(0, np.nan)
    )

    # Log transformation
    data["log_annual_inc"] = np.log1p(
        data["annual_inc"].clip(lower=0)
    )

    # Remove original date column
    data = data.drop(columns=["issue_d"])

    return data


# Demonstrate feature creation on training data
df_features = create_features(X_train.head(5))

print("New Features Created:")

display(
    df_features[
        [
            "issue_year",
            "issue_month",
            "income_loan_ratio",
            "log_annual_inc"
        ]
    ]
)

New Features Created:


,issue_year,issue_month,income_loan_ratio,log_annual_inc
1329991,2015,1,3.611111,11.082158
1936670,2014,10,6.000000,9.798183
1245871,2015,4,36.750000,11.205054
1280004,2015,3,2.791667,11.112463
1970057,2014,7,6.600000,10.404293


In [18]:
# Numeric columns
numeric_cols = [
    "loan_amnt",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "issue_year",
    "issue_month",
    "income_loan_ratio",
    "log_annual_inc"
]

# Categorical columns excluding grade
categorical_cols = [
    "term",
    "emp_length",
    "home_ownership",
    "verification_status",
    "purpose",
    "addr_state"
]

# Numeric Pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical Pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Ordinal Pipeline for Grade
ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["A", "B", "C", "D", "E", "F", "G"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
    ("ord", ordinal_pipeline, ["grade"])
])

print("Numeric Pipeline Created")
print("Categorical Pipeline Created")
print("Ordinal Pipeline Created")
print("\nColumnTransformer Created Successfully")

Numeric Pipeline Created
Categorical Pipeline Created
Ordinal Pipeline Created

ColumnTransformer Created Successfully


In [19]:
# Complete Machine Learning Pipeline
model_pipeline = Pipeline([
    ("feature_creation", FunctionTransformer(
        create_features,
        validate=False
    )),

    ("preprocessing", preprocessor),

    ("model", LogisticRegression(
        max_iter=1000,
        solver="liblinear"
    ))
])

print("Feature Engineering Added")
print("Preprocessing Added")
print("Logistic Regression Added")

print("\nComplete Pipeline Created Successfully!")

Feature Engineering Added
Preprocessing Added
Logistic Regression Added

Complete Pipeline Created Successfully!


In [20]:
# 5-Fold Cross Validation
cv_scores = cross_val_score(
    model_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Cross Validation Scores:", cv_scores)

print(
    "\nPipeline CV Accuracy: {:.4f}".format(
        cv_scores.mean()
    )
)

print(
    "Standard Deviation: {:.4f}".format(
        cv_scores.std()
    )
)

Cross Validation Scores: [0.79902196 0.80007671 0.79760915 0.79907946 0.80093332]

Pipeline CV Accuracy: 0.7993
Standard Deviation: 0.0011


In [21]:
# Train the complete pipeline
model_pipeline.fit(X_train, y_train)

print("Model Pipeline Trained Successfully!")

Model Pipeline Trained Successfully!


In [22]:
# Save the fitted pipeline
joblib.dump(
    model_pipeline,
    "loan_prediction_pipeline.pkl"
)

print("Pipeline Saved Successfully!")

Pipeline Saved Successfully!


In [23]:
# Load the saved pipeline
loaded_pipeline = joblib.load(
    "loan_prediction_pipeline.pkl"
)

print("Pipeline Loaded Successfully!")

# Select 3 raw rows from test dataset
new_rows = X_test.head(3)

# Predict using loaded pipeline
predictions = loaded_pipeline.predict(new_rows)

print("\nPredictions for 3 New Rows:", predictions)

# Display actual and predicted values
results = pd.DataFrame({
    "Actual": y_test.head(3).values,
    "Predicted": predictions
})

display(results)

Pipeline Loaded Successfully!

Predictions for 3 New Rows: [0 0 0]


,Actual,Predicted
0,0,0
1,1,0
2,0,0


# Preprocessing, Features and Pipeline

## Project Overview
This task focuses on preprocessing the Lending Club loan dataset, handling missing values, encoding categorical features, creating new features, and building a complete Machine Learning pipeline using Scikit-learn.

## Dataset
- **Dataset:** `Data/loan.csv`
- **Target Column:** `loan_status`
- **Selected Loan Statuses:**
  - Fully Paid – 0
  - Charged Off – 1
- **Sampling:** 15% of the filtered dataset was used to reduce computation time.

## Tasks Performed

### 1. Missing Value Handling
- Used `SimpleImputer(strategy="median")` for numerical columns.
- Used `SimpleImputer(strategy="most_frequent")` for categorical columns.
- Demonstrated `KNNImputer` with 5 neighbors on a sample of numeric data.

### 2. Categorical Encoding
- Applied `OneHotEncoder` to categorical features.
- Applied `OrdinalEncoder` to the ordered `grade` column.
- Grade order: A, B, C, D, E, F, G.

### 3. Feature Engineering
Created the following new features:

| Feature | Description |
|---|---|
| `issue_year` | Extracted year from loan issue date |
| `issue_month` | Extracted month from loan issue date |
| `income_loan_ratio` | Annual income divided by loan amount |
| `log_annual_inc` | Log transformation of annual income |

### 4. Preprocessing Pipeline
Used `ColumnTransformer` to combine:
- Numerical pipeline: Median Imputation + Standard Scaling.
- Categorical pipeline: Most Frequent Imputation + One-Hot Encoding.
- Ordinal pipeline: Most Frequent Imputation + Ordinal Encoding.

### 5. Machine Learning Pipeline
Built a complete Scikit-learn Pipeline containing:
1. Feature Creation
2. ColumnTransformer Preprocessing
3. Logistic Regression Model

### 6. Cross Validation
- Applied 5-Fold Cross Validation.
- Evaluation metric: Accuracy.
- Reported mean accuracy and standard deviation.

### 7. Model Saving and Prediction
- Saved the trained pipeline using `joblib`.
- Loaded the saved pipeline successfully.
- Generated predictions for 3 new raw test rows.

## Files
- `task2.ipynb` – Complete implementation and outputs.
- `loan_prediction_pipeline.pkl` – Saved trained Machine Learning pipeline.

## Conclusion
Successfully implemented a complete preprocessing and Machine Learning pipeline for the Lending Club loan dataset. Missing values were handled, categorical features were encoded, new features were created, and Logistic Regression was evaluated using 5-Fold Cross Validation. The trained pipeline was saved and loaded using Joblib to